# INFO 5304 Project - Phase 3: Preregistration

**Team:** Tara Munjuluri, Sahithya Swaminathan, Colette D'Costa

---

**Primary dataset used:** `phase1_predictors_complete.csv` (2019–2024, ~872 rows, one row per country-year with all six predictors present)

**Secondary dataset used:** `phase1_analysis_ready.csv` (2011–2024, ~1,969 rows, ladder score and rank only for years before 2019)

**Columns available in `phase1_predictors_complete.csv`:**
- `country` — country name (string)
- `year` — survey year (integer, 2019–2024)
- `rank` — happiness rank that year (integer)
- `ladder_score` — happiness score (numeric, ~2.4–8.0)
- `gdp_per_capita` — log GDP per capita (numeric)
- `social_support` — perceived social support (numeric)
- `healthy_life_expectancy` — healthy life expectancy at birth (numeric)
- `freedom_to_make_life_choices` — perceived freedom (numeric)
- `generosity` — self-reported generosity (numeric)
- `perceptions_of_corruption` — perceived low corruption (numeric; higher = less corrupt)

No additional data collection or cleaning is required to run the analyses below. The dataset is already in a tidy, analysis-ready format.

---

## Research Questions (from Phase 1)

1. Which country-level factors are most strongly associated with higher happiness scores?
2. Which countries outperform or underperform relative to their economic resources?
3. How have disparities in happiness-related predictors changed over time?

The four preregistered hypotheses below map directly onto these research questions.

---
## Hypothesis 1: GDP per capita is the single strongest predictor of happiness

**Hypothesis:** Among the six predictor variables (`gdp_per_capita`, `social_support`, `healthy_life_expectancy`, `freedom_to_make_life_choices`, `generosity`, `perceptions_of_corruption`), GDP per capita will have the largest standardized regression coefficient in a multiple linear regression predicting `ladder_score`, indicating it is the dominant driver of happiness across countries.

**Rationale:** Economic prosperity is widely cited as a foundation for meeting basic needs and reducing stress. While other social factors matter, we expect GDP to explain the greatest share of variance in happiness once all predictors are included simultaneously.

**Analysis:**

Using `phase1_predictors_complete.csv`, fit a multiple linear regression (OLS) where:
- **Outcome variable:** `ladder_score` (continuous numeric)
- **Predictor variables:** all six — `gdp_per_capita`, `social_support`, `healthy_life_expectancy`, `freedom_to_make_life_choices`, `generosity`, `perceptions_of_corruption` (all continuous numeric)

Before fitting, standardize all six predictors (subtract mean, divide by standard deviation) so that regression coefficients are directly comparable in magnitude. Use `sklearn.preprocessing.StandardScaler` or equivalent.

The model is:

$$\widehat{\text{ladder\_score}} = \beta_0 + \beta_1 \cdot \text{gdp\_std} + \beta_2 \cdot \text{social\_std} + \beta_3 \cdot \text{hle\_std} + \beta_4 \cdot \text{freedom\_std} + \beta_5 \cdot \text{generosity\_std} + \beta_6 \cdot \text{corruption\_std}$$

**Evaluation criteria:**
- Significance level: **α = 0.05**
- **Primary test:** Compare the absolute values of all six standardized β coefficients. The hypothesis is supported if $|\hat{\beta}_{\text{gdp}}|$ is strictly the largest among the six.
- **Secondary test:** Test whether $\beta_{\text{gdp}} > 0$ using a one-sided t-test at α = 0.05 (i.e., the coefficient is both the largest *and* positive).
- Report the full regression table (coefficients, standard errors, t-statistics, p-values, 95% CIs) and the model's R².
- Include a bar chart of standardized coefficients with confidence intervals for visual comparison.

**Standard library imports needed:** `pandas`, `numpy`, `sklearn.preprocessing.StandardScaler`, `statsmodels.api` (for OLS with full inference table), `matplotlib` / `seaborn`.

---
## Hypothesis 2: Countries with higher social support outperform their GDP-predicted happiness

**Hypothesis:** After controlling for GDP per capita, countries with above-median `social_support` will have systematically positive residuals from a GDP-only regression — meaning social support helps countries "punch above their economic weight" in terms of happiness.

**Rationale:** Countries like Costa Rica and Bhutan are frequently cited as happiness outliers relative to their income levels. We hypothesize that strong social networks explain this gap. If true, residuals from a GDP-only model will differ significantly by social-support group.

**Analysis:**

Using `phase1_predictors_complete.csv`:

**Step 1 — Fit a GDP-only baseline model.**  
Run a simple OLS regression:
$$\widehat{\text{ladder\_score}} = \beta_0 + \beta_1 \cdot \text{gdp\_per\_capita}$$
Compute residuals: $e_i = \text{ladder\_score}_i - \widehat{\text{ladder\_score}}_i$

**Step 2 — Create a binary grouping variable.**  
Calculate the median of `social_support` across all rows in `phase1_predictors_complete.csv`. Assign each row to one of two groups:
- `high_social` — `social_support` ≥ median
- `low_social` — `social_support` < median

**Step 3 — Compare residuals across groups.**  
Run an independent-samples two-sided t-test (`scipy.stats.ttest_ind`) comparing the residuals of the `high_social` group versus the `low_social` group.
- **Null hypothesis H₀:** Mean residual is the same for high- and low-social-support countries (μ_high = μ_low)
- **Alternative hypothesis H₁:** The means differ (μ_high ≠ μ_low)
- **Significance level:** α = 0.05
- Verify that Levene's test for equal variances is non-significant before using a pooled t-test; if significant, use Welch's t-test instead.

**Visualization:** Produce a boxplot of residuals split by `high_social` vs. `low_social`, overlaid with individual data points (strip plot). Annotate with the t-statistic and p-value.

**Step 4 — Identify top over- and under-performers.**  
List the 10 countries with the largest positive residuals and 10 with the largest negative residuals (averaged across years), to contextualize which specific countries are driving the effect.

**Standard library imports needed:** `pandas`, `numpy`, `scipy.stats` (`ttest_ind`, `levene`), `statsmodels.api`, `matplotlib` / `seaborn`.

---
## Hypothesis 3: The distribution of happiness scores differs significantly across world regions

**Hypothesis:** Countries in Western Europe will have a significantly higher mean `ladder_score` than countries in Sub-Saharan Africa, even after accounting for the panel structure of the data (multiple years per country).

**Rationale:** Regional clustering in world happiness is visually apparent in WHR reports. We preregister a formal test of whether the happiness gap between the highest- and lowest-ranked regions is statistically meaningful using a one-way ANOVA across all regions represented in the dataset.

**Data preparation note:**

`phase1_predictors_complete.csv` does not include a `region` column. A region mapping file will be constructed by merging country names with the UN macro-region classifications available from the WHR supplementary materials (or from the `countryinfo` Python package). Specifically:
1. Install `countryinfo` (`pip install countryinfo`) or manually create a lookup dictionary from [WHR 2024 Appendix Table A1](https://happiness-report.s3.amazonaws.com/2024/DataForFigure2.1WHR2024.xls) which includes region labels for each country.
2. Left-join region labels onto `phase1_predictors_complete.csv` on the `country` column.
3. Drop rows where region could not be matched (expected: fewer than 5 countries with non-standard names).

The resulting dataset will have a `region` column with categories such as: Western Europe, North America, Latin America & Caribbean, East Asia, Southeast Asia, South Asia, Central & Eastern Europe, Middle East & North Africa, Sub-Saharan Africa, Commonwealth of Independent States.

**Analysis:**

**Step 1 — One-way ANOVA.**  
For the region-merged dataset (one row per country-year), run a one-way ANOVA (`scipy.stats.f_oneway`) with:
- **Grouping variable:** `region` (categorical, 8–10 levels)
- **Outcome variable:** `ladder_score` (continuous numeric)
- **Null hypothesis H₀:** Mean `ladder_score` is the same across all regions.
- **Alternative hypothesis H₁:** At least one region's mean `ladder_score` differs from the others.
- **Significance level:** α = 0.05

**Step 2 — Post-hoc pairwise comparisons.**  
If the ANOVA is significant, run Tukey's HSD post-hoc test (`statsmodels.stats.multicomp.pairwise_tukeyhsd`) to identify which specific pairs of regions differ significantly. Report all pairwise mean differences and adjusted p-values.

**Step 3 — Focused comparison.**  
Regardless of overall ANOVA result, directly compare the mean `ladder_score` of Western Europe vs. Sub-Saharan Africa using an independent-samples two-sided t-test (α = 0.05), and report Cohen's d as an effect size measure.

**Visualization:** Produce a violin plot of `ladder_score` distributions by region, sorted by regional median, with individual country-year points overlaid.

**Standard library imports needed:** `pandas`, `numpy`, `scipy.stats.f_oneway`, `statsmodels.stats.multicomp.pairwise_tukeyhsd`, `matplotlib` / `seaborn`.

---
## Hypothesis 4: Happiness score disparities across countries have narrowed between 2019 and 2024

**Hypothesis:** The variance (spread) of `ladder_score` across countries has decreased from 2019 to 2024, indicating a convergence in global well-being over the period captured in `phase1_predictors_complete.csv`.

**Rationale:** Global events like the COVID-19 pandemic (2020–2021) may have temporarily compressed happiness scores by disproportionately lowering high-happiness countries, or widened the gap by devastating low-income nations. We preregister a formal test of whether variance changed over this six-year window.

**Analysis:**

Using `phase1_predictors_complete.csv`, subset to only the years 2019 and 2024 (the first and last available years) to compare endpoints. Restrict to countries present in **both** 2019 and 2024 to create a balanced panel (use `groupby('country').filter(lambda x: set([2019, 2024]).issubset(set(x['year'])))`).

**Step 1 — Descriptive statistics.**  
Compute the variance, standard deviation, and interquartile range of `ladder_score` separately for 2019 and 2024 within the balanced panel. Report the number of countries in the panel.

**Step 2 — Test for equality of variances.**  
Run Levene's test (`scipy.stats.levene`) comparing the distribution of `ladder_score` in 2019 vs. 2024:
- **Null hypothesis H₀:** The variance of `ladder_score` is the same in 2019 and 2024 (σ²_2019 = σ²_2024).
- **Alternative hypothesis H₁:** The variance differs (σ²_2019 ≠ σ²_2024).
- **Significance level:** α = 0.05

Additionally, run Bartlett's test (`scipy.stats.bartlett`) as a secondary check and report both results.

**Step 3 — Year-by-year trend.**  
Rather than comparing only 2019 and 2024, compute the standard deviation of `ladder_score` across all countries **for each year** from 2019 to 2024. Plot this as a line chart (x = year, y = std dev) to visualize whether the trend is monotonic or non-linear (e.g., a dip during 2020–2021 due to COVID). Fit a simple OLS regression of `std_dev ~ year` and report whether the slope β_year is significantly different from zero at α = 0.05.

**Standard library imports needed:** `pandas`, `numpy`, `scipy.stats` (`levene`, `bartlett`), `statsmodels.api`, `matplotlib` / `seaborn`.

---
## Summary Table

| # | Hypothesis | Analysis Type | Dataset | Significance Level |
|---|---|---|---|---|
| 1 | GDP per capita is the strongest predictor of happiness | Multiple OLS regression (standardized coefficients) | `phase1_predictors_complete.csv` | α = 0.05 |
| 2 | High social support → positive happiness residuals above GDP prediction | GDP-only OLS residuals + independent t-test | `phase1_predictors_complete.csv` | α = 0.05 |
| 3 | Happiness scores differ significantly across world regions | One-way ANOVA + Tukey HSD post-hoc | `phase1_predictors_complete.csv` + region merge | α = 0.05 |
| 4 | Cross-country happiness variance has narrowed from 2019 to 2024 | Levene's / Bartlett's test + trend regression | `phase1_predictors_complete.csv` | α = 0.05 |

> **Commitment statement:** We commit to reporting the results of all four analyses in the final report, regardless of whether the results are statistically significant or in the direction hypothesized above. We have not run any of these analyses prior to preregistration.